Acknowledgement: Consulted with Claude for the framework of code. AI-generated code in this project is manually debugged and heavily modified. As a result, the majority of the code in this file is manually written.

In [ ]:
import sqlite3
import csv

def create_flights_table(conn):
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS flights (
        flight_id TEXT PRIMARY KEY,
        origin TEXT,
        destination TEXT,
        capacity INTEGER,
        seats_remaining INTEGER
    );
    """
    conn.execute(create_table_sql)
    conn.commit()

def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {table_name}")  # clear existing rows first

    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [
            (row["flight_id"], row["origin"], row["destination"], row["capacity"], row["seats_remaining"])
            for row in reader
        ]

    sql = f"""
        INSERT INTO {table_name} (flight_id, origin, destination, capacity, seats_remaining)
        VALUES (?, ?, ?, ?, ?)
    """
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name} table.")

if __name__ == "__main__":
    conn = sqlite3.connect("flights_database.db") # NOTE: Claude suggested saving to the database file (.db), so the data persists after the notebook session ends
    create_flights_table(conn)          # create the table first
    load_csv_to_table(conn, "flights.csv")
    conn.close()

Loaded 200 rows into flights table.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flights_database.db")

def query(sql):
    """Helper: run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

# 1. View all flights
print(query("SELECT * FROM flights"))

# 2. Flights with available seats
print(query("""
    SELECT flight_id, origin, destination, seats_remaining
    FROM flights
    WHERE seats_remaining > 0
"""))

# 3. Fully booked flights
print(query("""
    SELECT flight_id, origin, destination, seats_remaining
    FROM flights
    WHERE seats_remaining = 0
"""))


    flight_id     source destination  capacity  seats_remaining
0         PA0    Calgary   Vancouver       150                6
1         PA1    Toronto      Ottawa       175               35
2         PA2    Toronto      Ottawa       250              108
3         PA3  Vancouver     Toronto       150               55
4         PA4     Ottawa    Montreal       250               50
..        ...        ...         ...       ...              ...
195     PA195    Calgary   Vancouver       150              141
196     PA196    Toronto     Calgary       225               26
197     PA197   Montreal     Calgary       225              122
198     PA198    Calgary    Montreal       175               13
199     PA199   Montreal      Ottawa       150               16

[200 rows x 5 columns]
    flight_id     source destination  seats_remaining
0         PA0    Calgary   Vancouver                6
1         PA1    Toronto      Ottawa               35
2         PA2    Toronto      Ottawa          

For the purpose of proving the concept of our table schema, I will use some mock data directly assigned to variables

In [1]:
import sqlite3
import csv
import uuid
import pandas as pd

In [2]:
def create_flights_table(conn):
    conn.execute("DROP TABLE IF EXISTS flights")  # remove old schema
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS flights (
        flight_uuid TEXT PRIMARY KEY,
        flight_number TEXT,
        origin TEXT NOT NULL,
        destination TEXT NOT NULL,
        capacity INTEGER,
        seats_remaining INTEGER
    );
    """
    conn.execute(create_table_sql)
    conn.commit()


In [3]:
def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {table_name}")

    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [
            (
                str(uuid.uuid4()),          # generate a fresh UUID per row
                row["flight_number"],
                row["origin"],
                row["destination"],
                row["capacity"],
                row["seats_remaining"],
            )
            for row in reader
        ]

    sql = f"""
        INSERT INTO {table_name} (flight_uuid, flight_number, origin, destination, capacity, seats_remaining)
        VALUES (?, ?, ?, ?, ?, ?)
    """
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name} table.")


In [4]:
if __name__ == "__main__":
    conn = sqlite3.connect("flights_database.db")
    try:
        create_flights_table(conn)
        load_csv_to_table(conn, "flights.csv")
    finally:
        print("\"flights\" table created and loaded successfully.")
        # conn.close()


Loaded 200 rows into flights table.
"flights" table created and loaded successfully.


In [5]:
def query(sql):
    """Helper: run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)


In [6]:
# 1. View all flights
print(query("SELECT * FROM flights"))


                              flight_uuid flight_number     origin  \
0    26e931fe-28fb-4cc2-981a-31de0f15aebe           PA0    Calgary   
1    32ba3cca-28dc-4f80-a8f0-35d4e9f5b0ec           PA1    Toronto   
2    ce5a7e92-b6fe-4b3d-b1f4-4a8c74c64113           PA2    Toronto   
3    74649634-9664-4097-a335-a1ce7a52799b           PA3  Vancouver   
4    ff1995db-67da-46d2-90a2-6815452a59bd           PA4     Ottawa   
..                                    ...           ...        ...   
195  766e3922-efc4-4504-90cd-1398154da7e8         PA195    Calgary   
196  feb69f5e-f9c0-4d72-9d9a-15096c80b1cd         PA196    Toronto   
197  a82d744f-4fea-4fc3-8015-791846cc0fd6         PA197   Montreal   
198  fe6ba17c-42b8-423d-bbec-b6dbe67aa477         PA198    Calgary   
199  9d63433f-93ef-4f76-b338-40448a122f1d         PA199   Montreal   

    destination  capacity  seats_remaining  
0     Vancouver       150                6  
1        Ottawa       175               35  
2        Ottawa       25

In [7]:
# 2. Flights with available seats
print(query("""
    SELECT flight_number, origin, destination, capacity, seats_remaining
    FROM flights
    WHERE seats_remaining > 0
"""))


    flight_number     origin destination  capacity  seats_remaining
0             PA0    Calgary   Vancouver       150                6
1             PA1    Toronto      Ottawa       175               35
2             PA2    Toronto      Ottawa       250              108
3             PA3  Vancouver     Toronto       150               55
4             PA4     Ottawa    Montreal       250               50
..            ...        ...         ...       ...              ...
193         PA195    Calgary   Vancouver       150              141
194         PA196    Toronto     Calgary       225               26
195         PA197   Montreal     Calgary       225              122
196         PA198    Calgary    Montreal       175               13
197         PA199   Montreal      Ottawa       150               16

[198 rows x 5 columns]


In [8]:
# 3. Fully booked flights
print(query("""
    SELECT flight_number, origin, destination, capacity, seats_remaining
    FROM flights
    WHERE seats_remaining = 0
"""))


  flight_number   origin destination  capacity  seats_remaining
0          PA33  Calgary   Vancouver       225                0
1         PA110   Ottawa   Vancouver       225                0


Now we will create the routes table:

In [9]:
def create_routes_table(conn):
    conn.execute("DROP TABLE IF EXISTS routes")  # remove old schema
    # NOTE: There could be multiple routes with the same origin and destination (i.e. multiple AC flights from YVR to YYZ every day)
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS routes (
        route_uuid TEXT PRIMARY KEY,
        origin TEXT NOT NULL,
        destination TEXT NOT NULL,
        base_fare DOUBLE,
        UNIQUE(origin, destination)   -- prevents duplicate route entries for the same O-D pair
    );
    """
    conn.execute(create_table_sql)
    conn.commit()


In [10]:
def load_data(conn, csv_path):
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        flight_rows = list(reader)

    # Unique routes
    seen = set()
    route_insert_rows = []
    for row in flight_rows:
        key = (row["origin"], row["destination"])
        if key not in seen:
            seen.add(key)
            route_insert_rows.append((str(uuid.uuid4()), row["origin"], row["destination"], None))

    cursor = conn.cursor()
    cursor.executemany(
        "INSERT INTO routes (route_uuid, origin, destination, base_fare) VALUES (?, ?, ?, ?)",
        route_insert_rows
    )

    # Flights — no route_uuid lookup needed anymore
    flight_insert_rows = [
        (str(uuid.uuid4()), row["flight_number"], row["origin"], row["destination"], row["capacity"], row["seats_remaining"])
        for row in flight_rows
    ]
    cursor.executemany(
        "INSERT INTO flights (flight_uuid, flight_number, origin, destination, capacity, seats_remaining) VALUES (?, ?, ?, ?, ?, ?)",
        flight_insert_rows
    )
    conn.commit()
    print(f"Loaded {len(route_insert_rows)} routes and {len(flight_insert_rows)} flights.")